### Prepare dialogue 

In [1]:
import pandas as pd

with open("../../own_script/dialogue_3/dialogue_3.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

df = pd.DataFrame({
    "utterance_id": range(1, len(lines)+1),
    "text": lines,
})
df.head()

,utterance_id,text
0,1,I've been feeling really overwhelmed lately. M...
1,2,"When I feel the palpitations, my heart starts ..."
2,3,"Well, the last time I went to the doctor, they..."
3,4,I guess if I could really believe that my hear...
4,5,"I think trying those techniques could help, bu..."


### Load text/VAD model

In [2]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


c:\Users\Legion 5 Pro\.conda\envs\w2v2vad\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.6.0+cu126
transformers: 5.3.0


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40162.69it/s]
BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Predict function

In [5]:
import numpy as np

def predict_vad(texts):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    # logits shape: [batch, 3] = [V, A, D]
    vad = out.logits.cpu().numpy()
    return vad  # np.array [batch,3]


### Check value range of vac-bert

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained("RobroKools/vad-bert")
model = AutoModelForSequenceClassification.from_pretrained("RobroKools/vad-bert").to(device)
model.eval()

samples = [
    "I feel terrible and hopeless.",
    "I feel completely neutral.",
    "I feel amazing and so happy!",
]

for s in samples:
    inputs = tok(s, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().cpu().tolist()
    print(s, "-> VAD:", out)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 44394.69it/s]
BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I feel terrible and hopeless. -> VAD: [1.762251615524292, 3.2708561420440674, 2.467195987701416]
I feel completely neutral. -> VAD: [2.781001091003418, 2.89453125, 3.203059196472168]
I feel amazing and so happy! -> VAD: [4.503336429595947, 4.0434064865112305, 3.532799005508423]


In [14]:
samples = [
    "I want to die. I hate everything.",
    "I feel completely empty and numb.",
    "This is fine.",
    "I'm a little annoyed.",
    "I'm so excited I can't stop screaming!",
    "I feel calm, peaceful, and relaxed.",
]

vals = []
for s in samples:
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().cpu().tolist()  # [V,A,D]
    print(s, "->", out)
    vals.append(out)

import numpy as np
vals = np.array(vals)
print("Valence range:", vals[:,0].min(), vals[:,0].max())
print("Arousal range:", vals[:,1].min(), vals[:,1].max())
print("Dominance range:", vals[:,2].min(), vals[:,2].max())


I want to die. I hate everything. -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
I feel completely empty and numb. -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
This is fine. -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
I'm a little annoyed. -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
I'm so excited I can't stop screaming! -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
I feel calm, peaceful, and relaxed. -> [1.4248106479644775, 3.8955318927764893, 2.7246105670928955]
Valence range: 1.4248106479644775 1.4248106479644775
Arousal range: 3.8955318927764893 3.8955318927764893
Dominance range: 2.7246105670928955 2.7246105670928955


### Test on dialogue 2 and save into dataframe

In [15]:
vad = predict_vad(df["text"].tolist())
df["valence_text"] = vad[:, 0]
df["arousal_text"] = vad[:, 1]
df["dominance_text"] = vad[:, 2]

df.head()

,utterance_id,text,valence_text,arousal_text,dominance_text
0,1,I've been feeling really overwhelmed lately. M...,2.097536,3.681398,2.381162
1,2,"When I feel the palpitations, my heart starts ...",2.330700,3.645309,2.423089
2,3,"Well, the last time I went to the doctor, they...",2.717745,3.235296,2.625666
3,4,I guess if I could really believe that my hear...,2.622219,3.526547,2.657344
4,5,"I think trying those techniques could help, bu...",2.975230,3.375428,3.043380


### Check with own dataset

In [16]:
import numpy as np

v = df["valence_text"].to_numpy()
a = df["arousal_text"].to_numpy()

print("Valence min/max:", v.min(), v.max())
print("Valence q1/median/q3:", np.quantile(v, [0.25, 0.5, 0.75]))

print("Arousal min/max:", a.min(), a.max())
print("Arousal q1/median/q3:", np.quantile(a, [0.25, 0.5, 0.75]))


Valence min/max: 2.0975363 3.274841
Valence q1/median/q3: [2.40357965 2.66998172 2.91085845]
Arousal min/max: 3.2352962 3.6813984
Arousal q1/median/q3: [3.41270411 3.52553928 3.61561877]


### Scale mismatch issue --> Normalization

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To solve this, I implement the normalization method to vac-bert scale to be compatible with wagner and the paper

In [17]:
import numpy as np

# กำหนดช่วง text สมมติเป็น 1..5
V_MIN, V_MAX = 1.0, 5.0

def to_minus1_1(x, xmin=V_MIN, xmax=V_MAX):
    return 2 * (x - xmin) / (xmax - xmin) - 1  # map [xmin,xmax] -> [-1,1]

df["valence_text_n"]  = to_minus1_1(df["valence_text"])
df["arousal_text_n"]  = to_minus1_1(df["arousal_text"])
df["dominance_text_n"] = to_minus1_1(df["dominance_text"])


In [18]:
df

,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n
0,1,I've been feeling really overwhelmed lately. M...,2.097536,3.681398,2.381162,-0.451232,0.340699,-0.309419
1,2,"When I feel the palpitations, my heart starts ...",2.330700,3.645309,2.423089,-0.334650,0.322655,-0.288456
2,3,"Well, the last time I went to the doctor, they...",2.717745,3.235296,2.625666,-0.141128,0.117648,-0.187167
3,4,I guess if I could really believe that my hear...,2.622219,3.526547,2.657344,-0.188891,0.263273,-0.171328
4,5,"I think trying those techniques could help, bu...",2.975230,3.375428,3.043380,-0.012385,0.187714,0.021690
5,6,That sounds like a good idea! I think having a...,3.274841,3.524532,3.352258,0.137421,0.262266,0.176129


### Assert mutual absolute scale --> Check by revert back to its previous form 

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To check this, I implement the revert method to vac-bert scale and assert it to contain the same value for original scale then there is unchanged in absolute meaning in value

- The output must be 0

In [19]:
# map 1..5 -> -1..1
def one5_to_minus1_1(x, xmin=1.0, xmax=5.0):
    x01 = (x - xmin) / (xmax - xmin)
    return 2*x01 - 1

def minus1_1_to_one5(y, xmin=1.0, xmax=5.0):
    x01 = (y + 1) / 2
    return x01 * (xmax - xmin) + xmin

diff = df["arousal_text"] - minus1_1_to_one5(df["arousal_text_n"])
print(diff.abs().max())   # ควร ~ 0 (มีแค่ numerical noise ระดับ 1e-7)


0.0


### Check rank & order value

- The output must be 1

In [20]:
# index ของค่า arousal สูงสุด ก่อนและหลัง normalize ต้องเป็นอันเดียวกัน
orig_argmax = df["arousal_text"].idxmax()
norm_argmax = df["arousal_text_n"].idxmax()
print(orig_argmax, norm_argmax)

# หรือ correlation ระหว่างค่าเดิมกับค่าที่ normalize ควร = 1
df[["arousal_text", "arousal_text_n"]].corr()


0 0


,arousal_text,arousal_text_n
arousal_text,1.0,1.0
arousal_text_n,1.0,1.0


### Check distribution

In [21]:
print(df["arousal_text"].describe())
print(df["arousal_text_n"].describe())


count    6.000000
mean     3.498085
std      0.167864
min      3.235296
25%      3.412704
50%      3.525539
75%      3.615619
max      3.681398
Name: arousal_text, dtype: float64
count    6.000000
mean     0.249043
std      0.083932
min      0.117648
25%      0.206352
50%      0.262770
75%      0.307809
max      0.340699
Name: arousal_text_n, dtype: float64


### Save to .csv format

In [23]:
df.to_csv("../../own_script/dialogue_3/dialogue_3_vad_text.csv", index=False)